# 02 - Preprocessing

## Overview

Clean data, handle infinite values, scale features, encode labels, and save to Parquet for faster loading.

**Output**: Parquet files ready for FL training

In [1]:
# Imports
import numpy as np
import pandas as pd
import os
import json
import joblib
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split

import sys
sys.path.insert(0, '/home/willtanoe/Documents/fl-xgb-thesis')
from src.config import SEED, TRAIN_SUBSAMPLE_RATIO

# Paths
DATA_DIR = "/home/willtanoe/Documents/CICIOT23"
RESULTS_DIR = "/home/willtanoe/Documents/fl-xgb-thesis/results"
PREPROCESSED_DIR = os.path.join(RESULTS_DIR, "preprocessed")

os.makedirs(PREPROCESSED_DIR, exist_ok=True)
os.makedirs(os.path.join(RESULTS_DIR, "figures"), exist_ok=True)
os.makedirs(os.path.join(RESULTS_DIR, "logs"), exist_ok=True)

# Load from EDA summary
print(f"Preprocessed dir: {PREPROCESSED_DIR}")
print(f"Subsample ratio: {TRAIN_SUBSAMPLE_RATIO}")

Preprocessed dir: /home/willtanoe/Documents/fl-xgb-thesis/results/preprocessed
Subsample ratio: 0.25


## 1. Load Data

In [2]:
print("Loading data from CSV...")
train = pd.read_csv(os.path.join(DATA_DIR, "train", "train.csv"))
val = pd.read_csv(os.path.join(DATA_DIR, "validation", "validation.csv"))
test = pd.read_csv(os.path.join(DATA_DIR, "test", "test.csv"))

print(f"Original train shape: {train.shape}")
print(f"Original val shape: {val.shape}")
print(f"Original test shape: {test.shape}")

Loading data from CSV...
Original train shape: (5491971, 47)
Original val shape: (1176851, 47)
Original test shape: (1176851, 47)


## 2. Subsample Training Data (25%)

In [3]:
# Stratified subsample untuk maintain class distribution
if TRAIN_SUBSAMPLE_RATIO < 1.0:
    print(f"Subsampling train to {TRAIN_SUBSAMPLE_RATIO*100:.0f}% with stratification...")
    
    train, _ = train_test_split(
        train,
        train_size=TRAIN_SUBSAMPLE_RATIO,
        stratify=train['label'],
        random_state=SEED
    )
    print(f"Subsampled train shape: {train.shape}")
else:
    print("Using full training data.")

Subsampling train to 25% with stratification...
Subsampled train shape: (1372992, 47)


## 3. Handle Infinite Values

In [4]:
def handle_infinities(df, label_col='label'):
    """Replace inf/-inf with NaN, then fill with median."""
    df = df.copy()
    feature_cols = [c for c in df.columns if c != label_col]
    
    for col in feature_cols:
        if df[col].dtype in [np.float64, np.float32]:
            # Count inf values
            inf_count = np.isinf(df[col]).sum()
            if inf_count > 0:
                print(f"  {col}: {inf_count} inf values")
                df[col] = df[col].replace([np.inf, -np.inf], np.nan)
                df[col] = df[col].fillna(df[col].median())
    
    return df

print("Handling infinities in train...")
train = handle_infinities(train)
print("Handling infinities in val...")
val = handle_infinities(val)
print("Handling infinities in test...")
test = handle_infinities(test)

print("\nDone.")

Handling infinities in train...
Handling infinities in val...
Handling infinities in test...

Done.


## 4. Feature Selection (Remove Constant Columns)

In [5]:
# Find constant columns
feature_cols = [c for c in train.columns if c != 'label']
variances = train[feature_cols].var()
constant_cols = variances[variances == 0].index.tolist()

if constant_cols:
    print(f"Removing {len(constant_cols)} constant columns: {constant_cols}")
    feature_cols = [c for c in feature_cols if c not in constant_cols]
else:
    print("No constant columns found.")

print(f"\nTotal features after removal: {len(feature_cols)}")

Removing 3 constant columns: ['Telnet', 'SMTP', 'IRC']

Total features after removal: 43


## 5. Encode Labels

In [6]:
print("Fitting LabelEncoder on train labels...")
le = LabelEncoder()
le.fit(train['label'])

# Encode all sets
train['label_encoded'] = le.transform(train['label'])
val['label_encoded'] = le.transform(val['label'])
test['label_encoded'] = le.transform(test['label'])

print(f"\nClasses ({len(le.classes_)}):")
for i, cls in enumerate(le.classes_):
    count = (train['label'] == cls).sum()
    print(f"  {i:2d}: {cls:30s} ({count:,} samples)")

# Save label encoder
joblib.dump(le, os.path.join(PREPROCESSED_DIR, "label_encoder.pkl"))
print(f"\nLabel encoder saved: {PREPROCESSED_DIR}/label_encoder.pkl")

Fitting LabelEncoder on train labels...

Classes (34):
   0: Backdoor_Malware               (98 samples)
   1: BenignTraffic                  (32,385 samples)
   2: BrowserHijacking               (166 samples)
   3: CommandInjection               (155 samples)
   4: DDoS-ACK_Fragmentation         (8,395 samples)
   5: DDoS-HTTP_Flood                (843 samples)
   6: DDoS-ICMP_Flood                (212,022 samples)
   7: DDoS-ICMP_Fragmentation        (13,262 samples)
   8: DDoS-PSHACK_Flood              (120,313 samples)
   9: DDoS-RSTFINFlood               (118,860 samples)
  10: DDoS-SYN_Flood                 (119,663 samples)
  11: DDoS-SlowLoris                 (689 samples)
  12: DDoS-SynonymousIP_Flood        (105,521 samples)
  13: DDoS-TCP_Flood                 (132,125 samples)
  14: DDoS-UDP_Flood                 (159,389 samples)
  15: DDoS-UDP_Fragmentation         (8,542 samples)
  16: DNS_Spoofing                   (5,304 samples)
  17: DictionaryBruteForce           (3

## 6. Prepare Feature Arrays

In [7]:
# Extract features and labels
X_train = train[feature_cols].values.astype(np.float32)
y_train = train['label_encoded'].values.astype(np.int32)

X_val = val[feature_cols].values.astype(np.float32)
y_val = val['label_encoded'].values.astype(np.int32)

X_test = test[feature_cols].values.astype(np.float32)
y_test = test['label_encoded'].values.astype(np.int32)

print(f"X_train shape: {X_train.shape}")
print(f"X_val shape: {X_val.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"\ny_train classes: {len(np.unique(y_train))}")
print(f"y_val classes: {len(np.unique(y_val))}")
print(f"y_test classes: {len(np.unique(y_test))}")

X_train shape: (1372992, 43)
X_val shape: (1176851, 43)
X_test shape: (1176851, 43)

y_train classes: 34
y_val classes: 34
y_test classes: 34


## 7. Scaling (StandardScaler)

In [8]:
print("Fitting StandardScaler on training data...")
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

print(f"Scaler fitted. Mean range: [{scaler.mean_.min():.4f}, {scaler.mean_.max():.4f}]")
print(f"Scaler std range: [{scaler.scale_.min():.4f}, {scaler.scale_.max():.4f}]")

# Save scaler
joblib.dump(scaler, os.path.join(PREPROCESSED_DIR, "scaler.pkl"))
print(f"Scaler saved: {PREPROCESSED_DIR}/scaler.pkl")

Fitting StandardScaler on training data...
Scaler fitted. Mean range: [0.0000, 83184832.0132]
Scaler std range: [0.0007, 17061470.1902]
Scaler saved: /home/willtanoe/Documents/fl-xgb-thesis/results/preprocessed/scaler.pkl


## 8. Save to Parquet

In [9]:
print("Saving to Parquet...")

# Save train
train_df = pd.DataFrame(X_train, columns=feature_cols)
train_df['label'] = y_train
train_df.to_parquet(os.path.join(PREPROCESSED_DIR, "train.parquet"), index=False)
print(f"  train.parquet: {os.path.getsize(os.path.join(PREPROCESSED_DIR, 'train.parquet')) // (1024*1024)} MB")

# Save val
val_df = pd.DataFrame(X_val, columns=feature_cols)
val_df['label'] = y_val
val_df.to_parquet(os.path.join(PREPROCESSED_DIR, "val.parquet"), index=False)
print(f"  val.parquet: {os.path.getsize(os.path.join(PREPROCESSED_DIR, 'val.parquet')) // (1024*1024)} MB")

# Save test
test_df = pd.DataFrame(X_test, columns=feature_cols)
test_df['label'] = y_test
test_df.to_parquet(os.path.join(PREPROCESSED_DIR, "test.parquet"), index=False)
print(f"  test.parquet: {os.path.getsize(os.path.join(PREPROCESSED_DIR, 'test.parquet')) // (1024*1024)} MB")

Saving to Parquet...
  train.parquet: 49 MB
  val.parquet: 43 MB
  test.parquet: 42 MB


## 9. Compute Class Weights

In [10]:
from sklearn.utils.class_weight import compute_class_weight

# Compute class weights for imbalanced data
classes = np.unique(y_train)
class_weights = compute_class_weight('balanced', classes=classes, y=y_train)
class_weight_dict = dict(zip(classes, class_weights))

print("Class weights (top 10 by weight):")
sorted_weights = sorted(class_weight_dict.items(), key=lambda x: x[1], reverse=True)[:10]
for cls, weight in sorted_weights:
    print(f"  Class {cls:2d} ({le.classes_[cls]:25s}): {weight:.4f}")

# Save class weights
joblib.dump(class_weight_dict, os.path.join(PREPROCESSED_DIR, "class_weights.pkl"))
print(f"\nClass weights saved: {PREPROCESSED_DIR}/class_weights.pkl")

Class weights (top 10 by weight):
  Class 31 (Uploading_Attack         ): 1153.7748
  Class 28 (Recon-PingSweep          ): 651.3245
  Class  0 (Backdoor_Malware         ): 412.0624
  Class 33 (XSS                      ): 388.2896
  Class 30 (SqlInjection             ): 272.8521
  Class  3 (CommandInjection         ): 260.5298
  Class  2 (BrowserHijacking         ): 243.2658
  Class 17 (DictionaryBruteForce     ): 104.8886
  Class 11 (DDoS-SlowLoris           ): 58.6097
  Class  5 (DDoS-HTTP_Flood          ): 47.9029

Class weights saved: /home/willtanoe/Documents/fl-xgb-thesis/results/preprocessed/class_weights.pkl


## 10. Save Metadata

In [11]:
# Save preprocessing metadata
metadata = {
    'feature_cols': feature_cols,
    'constant_cols': constant_cols,
    'num_features': len(feature_cols),
    'num_classes': len(le.classes_),
    'classes': le.classes_.tolist(),
    'subsample_ratio': TRAIN_SUBSAMPLE_RATIO,
    'train_rows': int(len(X_train)),
    'val_rows': int(len(X_val)),
    'test_rows': int(len(X_test)),
    'seed': SEED
}

with open(os.path.join(PREPROCESSED_DIR, "metadata.json"), 'w') as f:
    json.dump(metadata, f, indent=2)

print("Preprocessing metadata saved!")
print(f"\nSummary:")
print(f"  Features: {len(feature_cols)}")
print(f"  Classes: {len(le.classes_)}")
print(f"  Train rows: {len(X_train):,}")
print(f"  Val rows: {len(X_val):,}")
print(f"  Test rows: {len(X_test):,}")

Preprocessing metadata saved!

Summary:
  Features: 43
  Classes: 34
  Train rows: 1,372,992
  Val rows: 1,176,851
  Test rows: 1,176,851


## Summary

- Replaced inf values with median
- Removed constant columns
- Label-encoded 36 classes
- Scaled features with StandardScaler
- Saved as Parquet files
- Computed class weights for imbalanced handling

**Next**: Proceed to `03_fl_setup.ipynb` for FL setup.